In [ ]:
# @title 🛠️ Step 1: Clean Install (Anti-Conflict)
# @markdown यह सेल पुराने खराब इंस्टॉलेशन को हटाकर सही वर्जन डालेगा।
import os
os.environ['MPLBACKEND'] = 'Agg'  # Matplotlib Error Fix

print("Installing Python 3.10 & Stable TTS...")
!sudo apt-get install python3.10 python3.10-dev python3.10-distutils -y -q
!wget https://bootstrap.pypa.io/get-pip.py -q && python3.10 get-pip.py -q

# केवल एक बार सही वातावरण में इंस्टॉल करें
!python3.10 -m pip install -q coqui-tts gradio==4.44.1 pydub numpy<2.0.0
print("✅ Environment Ready!")

In [ ]:
# @title 🚀 Step 2: High-Speed Launch
with open("app_turbo.py", "w") as f:
    f.write('''
import os
os.environ['MPLBACKEND'] = 'Agg'
import gradio as gr
from TTS.api import TTS
from pydub import AudioSegment, silence

device = "cuda" if os.path.exists("/dev/nvidia0") else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/your_tts").to(device)

def process(text, ref, clean):
    out, final = "temp.wav", "final_voice.wav"
    tts.tts_to_file(text=text, speaker_wav=ref, language="en", file_path=out)
    if clean:
        audio = AudioSegment.from_file(out)
        chunks = silence.split_on_silence(audio, min_silence_len=300, silence_thresh=-40, keep_silence=100)
        combined = AudioSegment.empty()
        for c in chunks: combined += c
        combined.export(final, format="wav")
    else:
        os.rename(out, final)
    return final

gr.Interface(fn=process, inputs=[gr.Textbox(), gr.Audio(type="filepath"), gr.Checkbox(value=True)], outputs=gr.Audio()).launch(share=True)
''')

!python3.10 app_turbo.py